In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# binary class인 경우, BinaryclassClassificationEvaluator 사용

In [2]:
spark = SparkSession.builder.appName('16').getOrCreate()

In [3]:
data = spark.read.csv('C:/Users/User/Downloads/iris.csv',
                      header=True,inferSchema=True)

- `train/test` 분리

In [5]:
train_data, test_data = data.randomSplit([0.8,0.2],seed=42)

## 1. `VectorAssembler`

In [7]:
# inputCols: sepal_length, sepal_width, petal_length, petal_width
# outputCol: iris_features
va = VectorAssembler(
    inputCols=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
    outputCol='iris_features'
)

## 2. `MinMaxScaler`
- `MinMaxScaler(inputCol, outputCol, min, max)`
> - `inputCol`: 변환하고자 하는 column 이름
> - `outputCol`: 변환 후의 값이 저장되는 column 이름
> - `min, max`: 스케일링 후의 최소 최대 값(디폴트: `0.0, 1.0`)

In [8]:
scaler = MinMaxScaler(inputCol='iris_features',outputCol='iris_features_sc')

## 3. `MultilayerPerceptronClassifier` 학습
- `layers`: 각 layer를 구성하는 node의 수를 list로 지정 (첫번째 값은 반드시 입력 특징의 수와 동일해야 하며, 마지막 값은 분류되는 클래스의 수와 동일해야 함)
> - **특징의 수**(feature vector column의 list가 포함하고 있는 값의 수): `sepal_length, sepal_width, petal_length, petal_width` **총 4개**
> - **클래스의 수**: `0, 1, 2` **총 3개**
- `featuresCol, labelCol`: 특징 vector(`X`) column, `y` column
> - `featuresCol`: Scaling 후의 column 값을 사용, `iris_features_sc`
> - `labelCol`: `species`
- `predictionCol`: 모델이 예측한 분류 결과가 저장될 column의 이름
- `stepSize`(디폴트: 0.03): learning rate(학습률), 일반적으로 `0.01 ~ 0.1` 범위
- `maxIter`(디폴트: 100): epoch
- `blockSize`(디폴트: 128): batch size

In [14]:
mlp = MultilayerPerceptronClassifier(
    featuresCol='iris_features_sc',
    labelCol='species',
    predictionCol='mlp_predict',
    maxIter=50
)

## 4. `Pipeline`
- `Pipeline(stages=[a,b,c])`: stages에 list형태로 전처리부터 모델까지 순서대로 지정

In [15]:
mlp_pipeline = Pipeline(stages=[va,scaler,mlp])

## 5. `ParamGridBuilder`
- model의 파라미터를 여러 값으로 지정한 후, 최고 성능의 model을 찾음
- `ParamGridBuilder().addGrid(parameter, value list).addGrid()...build()`

In [19]:
# layer = [[4,5,4,3], [4,10,8,3]] (반드시 첫번째 값은 4, 마지막 값은 3으로 지정)
# stepSize = [0.03, 0.1]
param_grid = ParamGridBuilder().\
addGrid(mlp.layers,[[4,5,4,3], [4,10,3]]).\
addGrid(mlp.stepSize,[0.03,0.1]).\
build()

## 6. `CrossValidator`
- 주어진 데이터를 n개로 분리하여, 일부를 검증용 데이터로 활용하는 방식 (n개로 분리할 경우, 총 n번의 테스트를 거침)
- `CrossValidator()`
> - `estimator`: classification 및 regression 모델 또는 Pipeline
> - `estimatorParamMaps`: 일반적으로 `ParamGridBuilder`의 결과
> - `evaluator`: 성능평가 척도(`RegressionEvaluator`, `MulticlassClassificationEvaluator` 등)
> - `numFolds`: 분리할 데이터의 수 (n-fold)

In [20]:
mlp_eval = MulticlassClassificationEvaluator(predictionCol='mlp_predict',
                                             labelCol='species',
                                             metricName='accuracy')

In [21]:
cv = CrossValidator(estimator=mlp_pipeline,
                    estimatorParamMaps=param_grid,
                    evaluator=mlp_eval,
                    numFolds=3)

## 7. 일괄 학습 및 예측
- `CrossValidator.fit(data)`

In [22]:
cv_model = cv.fit(train_data)

In [23]:
# CrossValidatorModel을 반환
type(cv_model)

pyspark.ml.tuning.CrossValidatorModel

In [29]:
# CrossValidatorModel.transform(data): 최고 성능 모델에 대한 예측값을 포함한 DataFrame 반환
# iris_features(feature vector), iris_features_sc(스케일링 된 feature vector)
# mlp_predict(예측 결과)
predicted_species = cv_model.transform(test_data)
train_predicted_species = cv_model.transform(train_data)

In [27]:
predicted_species.show(3)

+------------+-----------+------------+-----------+-------+-----------------+--------------------+--------------------+--------------------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|species|    iris_features|    iris_features_sc|       rawPrediction|         probability|mlp_predict|
+------------+-----------+------------+-----------+-------+-----------------+--------------------+--------------------+--------------------+-----------+
|         4.4|        3.0|         1.3|        0.2|      0|[4.4,3.0,1.3,0.2]|[0.02777777777777...|[24.9685285436839...|[0.99999999999624...|        0.0|
|         4.6|        3.2|         1.4|        0.2|      0|[4.6,3.2,1.4,0.2]|[0.08333333333333...|[25.1110662524989...|[0.99999999999690...|        0.0|
|         4.6|        3.6|         1.0|        0.2|      0|[4.6,3.6,1.0,0.2]|[0.08333333333333...|[25.4206551771424...|[0.99999999999798...|        0.0|
+------------+-----------+------------+-----------+-------+-----------------+-----

In [30]:
print("Train acc: ", mlp_eval.evaluate(train_predicted_species))
print("Test acc : ", mlp_eval.evaluate(predicted_species))

Train acc:  0.984251968503937
Test acc :  1.0


## * `CrossValidator`의 `bestModel`

- `CrossValidator`의 `estimator`를 `Pipeline`으로 설정했으므로, `bestModel`도 `PipelineModel`
- `CrossValidator`의 `estimator`를 `MultilayerPerceptronClassifier`로 설정했다면, `bestModel`은 `MultilayerPerceptronClassificationModel`이 됨

In [32]:
print(type(cv_model))
print(type(cv_model.bestModel))

<class 'pyspark.ml.tuning.CrossValidatorModel'>
<class 'pyspark.ml.pipeline.PipelineModel'>


- 만약, `Pipeline` 전체가 아닌 `mlp`만 확인하고 싶다면, `PipelineModel`의 `stages`를 통해 선택할 수 있음
- 위의 경우
> - `stages[0]`: `VectorAssembler`
> - `stages[1]`: `MinMaxScaler`
> - `stages[2]`: `MultilayerPerceptronClassifier`

In [34]:
print(type(cv_model.bestModel.stages[-1]))

<class 'pyspark.ml.classification.MultilayerPerceptronClassificationModel'>


In [35]:
# 최고 성능의 파라미터 확인
print('최고 성능 layer 구성: ', cv_model.bestModel.stages[-1].getLayers())
print('최고 성능 learning rate: ', cv_model.bestModel.stages[-1].getStepSize())

최고 성능 layer 구성:  [4, 5, 4, 3]
최고 성능 learning rate:  0.03
